# Thí nghiệm KNN — Vietnamese SMS Spam Detection (Người 2)

Notebook này giải thích toàn bộ thí nghiệm KNN trên bộ dữ liệu SMS tiếng Việt chính thức
(`trannguyenthaituan251209/vietnamese_sms_dataset`), tuân thủ `docs/experiment_rules.md`:

1. **Dữ liệu**: `train.csv` (2.394 mẫu) dùng huấn luyện + tuning; `test.csv` (597 mẫu) **chỉ** dùng đánh giá cuối.
2. **Preprocessing**: chỉ dùng `common.preprocessing.clean_series` (chuẩn chung của cả 2 thành viên).
3. **Vectorization**: CountVectorizer và TF-IDF với `max_features=5000`, `ngram_range=(1,1)`; fit trên train, transform test.
4. **Tuning K**: 5-fold Stratified CV **chỉ trên `train.csv`** với `k ∈ {3, 5, 7, 9, 11, 15, 21}`, dùng `sklearn.pipeline.Pipeline` (vectorizer → KNN) chạy trên **văn bản** — mỗi fold tự fit vectorizer trên phần train của fold đó, **không rò rỉ dữ liệu** từ fold validation vào từ vựng / trọng số IDF.
5. **Đánh giá cuối**: Accuracy / Precision / Recall / F1 / Confusion Matrix + thời gian train/predict trên `test.csv` (một lần duy nhất).
6. **Error analysis**: 5 False Positive + 5 False Negative cho mỗi cấu hình.

Kết quả số liệu trùng khớp với `python scripts/run_knn.py` (cùng `RANDOM_STATE = 42`).

## 0. Setup

In [1]:
import sys
from pathlib import Path

# Tìm project root một cách chắc chắn: thư mục cha chứa config/settings.py.
_here = Path.cwd().resolve()
PROJECT_ROOT = next(p for p in [_here, *_here.parents] if (p / "config" / "settings.py").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Console Windows có thể dùng cp1252 — ép UTF-8 để in tiếng Việt.
for _stream in (sys.stdout, sys.stderr):
    if hasattr(_stream, "reconfigure"):
        _stream.reconfigure(encoding="utf-8", errors="replace")

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from config.settings import (
    LABEL_COLUMN, TEXT_COLUMN, TRAIN_PATH, TEST_PATH,
    KNN_K_VALUES, CV_FOLDS, RANDOM_STATE, MAX_FEATURES, NGRAM_RANGE,
)
from common.data_loader import load_sms_csv, dataset_summary
from common.preprocessing import clean_series
from common.feature_extraction import (
    build_count_vectorizer, build_tfidf_vectorizer, fit_transform_train_test,
)
from common.metrics import classification_metrics, timed_fit_predict, result_row
from models.knn.train_knn import build_knn
from models.knn.tune_knn import tune_k_cross_val, pick_best_k, build_knn_pipeline
from models.knn.evaluate_knn import (
    plot_confusion_matrix, plot_k_chart, _error_frame, write_error_analysis_md,
    RESULTS_DIR,
)
print("Project root:", PROJECT_ROOT)

Project root: C:\email_spam\.claude\worktrees\knn-finish


## 1. Load và kiểm tra dữ liệu (Checklist B)

- In số dòng train/test, kiểm tra cột `message`/`label`.
- Kiểm tra missing / duplicate, thống kê Ham/Spam.
- Đối chiếu với Người 1: cả hai đều đọc bằng `common.data_loader.load_sms_csv` nên số liệu phải giống hệt nhau.

In [2]:
train = load_sms_csv(TRAIN_PATH)
test = load_sms_csv(TEST_PATH)

summary_train = dataset_summary(train)
summary_test = dataset_summary(test)
print("TRAIN:", summary_train)
print("TEST :", summary_test)

pd.DataFrame([
    {"set": "train", **summary_train},
    {"set": "test", **summary_test},
])

TRAIN: {'rows': 2394, 'missing_message': 0, 'empty_message': 0, 'duplicates': 303, 'ham': 1754, 'spam': 640}
TEST : {'rows': 597, 'missing_message': 0, 'empty_message': 0, 'duplicates': 31, 'ham': 439, 'spam': 158}


,set,rows,missing_message,empty_message,duplicates,ham,spam
0,train,2394,0,0,303,1754,640
1,test,597,0,0,31,439,158


### Duplicate trong dữ liệu chính thức

Theo cặp `(message, label)`, dữ liệu **có duplicate**: **303 dòng trùng ở `train.csv`** và **31 dòng trùng ở `test.csv`** (không có missing/empty).

**Cách xử lý: giữ nguyên toàn bộ dữ liệu gốc.**

- Không tự xóa duplicate, không chia lại train/test — dataset chính thức là tài sản chung của nhóm, mọi thay đổi phải thống nhất với cả nhóm (`docs/TEAM_CHECKLIST.md`).
- Cả hai thành viên (Naive Bayes và KNN) đều dùng chung một bản dữ liệu nên việc so sánh vẫn công bằng.
- Lưu ý trung thực: một số tin nhắn xuất hiện ở cả train lẫn test (do duplicate), KNN có thể "nhận ra" láng giềng gần trùng lặp — đây là đặc điểm của dataset chính thức, không phải lỗi quy trình; số liệu báo cáo đúng như dữ liệu gốc.

In [3]:
train.head()

,message,label
0,"Ma OTP xac thuc GD la 066595, hieu luc [TIME]....",0
1,[TB] CMND đăng ký thuê bao của Quý khách đã hế...,0
2,[TB] Số điện thoại của Quý khách đã được xác t...,0
3,"Theo thông tư 08/TT-BKHCN của Bộ KH&CN, để đảm...",0
4,Xác thực NGAY để không bị TẠM NGỪNG dịch vụ da...,0


## 2. Preprocessing (Checklist C)

Chỉ dùng `common.preprocessing.clean_series`:
- missing → chuỗi rỗng, chuẩn hóa Unicode NFC, lowercase;
- URL → `__url__`, email → `__email__`;
- **bảo toàn** token ẩn danh `[PHONE]`, `[MONEY]`, `[DATE]`, `[TIME]`, ... (chuyển thành `__phone__`, `__money__`, ...);
- không xóa chữ số một cách máy móc (mã OTP, số tiền là tín hiệu spam mạnh).

Xem 10 mẫu trước/sau:

In [4]:
train_text = clean_series(train[TEXT_COLUMN])
test_text = clean_series(test[TEXT_COLUMN])
y_train = train[LABEL_COLUMN]
y_test = test[LABEL_COLUMN]

for raw, cleaned in list(zip(train[TEXT_COLUMN], train_text))[:10]:
    raw_s = " ".join(str(raw).split())[:100]
    print(f"RAW : {raw_s}")
    print(f"CLEAN: {cleaned[:100]}")
    print("-" * 80)

RAW : Ma OTP xac thuc GD la 066595, hieu luc [TIME]. Chi tiet GD: Chuyen khoan nhanh qua so TK, so tien [M
CLEAN: ma otp xac thuc gd la 066595 hieu luc __time__ chi tiet gd chuyen khoan nhanh qua so tk so tien __mo
--------------------------------------------------------------------------------
RAW : [TB] CMND đăng ký thuê bao của Quý khách đã hết hiệu lực từ [DATE] (theo quy định luật Căn cước). Qu
CLEAN: tb cmnd đăng ký thuê bao của quý khách đã hết hiệu lực từ __date__ theo quy định luật căn cước quý k
--------------------------------------------------------------------------------
RAW : [TB] Số điện thoại của Quý khách đã được xác thực thông tin theo quy định của Thông tư 08/[NUMBER]/ 
CLEAN: tb số điện thoại của quý khách đã được xác thực thông tin theo quy định của thông tư 08 __number__ t
--------------------------------------------------------------------------------
RAW : Theo thông tư 08/TT-BKHCN của Bộ KH&CN, để đảm bảo quyền lợi & tránh gián đoạn dịch vụ, QK cần cập n
CLEAN

## 3. Vectorization (Checklist D)

Hai cách biểu diễn văn bản, config chung từ `config/settings.py` (`MAX_FEATURES=5000`, `NGRAM_RANGE=(1,1)`).
Vectorizer chỉ **fit trên train**, test chỉ **transform** — chống data leakage.

In [5]:
shapes = {}
for name, builder in [("Count", build_count_vectorizer), ("TFIDF", build_tfidf_vectorizer)]:
    vec = builder()
    x_tr, x_te = fit_transform_train_test(vec, train_text, test_text)
    shapes[name] = (x_tr.shape, x_te.shape)
    print(f"{name:>5}: X_train = {x_tr.shape}, X_test = {x_te.shape}")

Count: X_train = (2394, 4346), X_test = (597, 4346)


TFIDF: X_train = (2394, 4346), X_test = (597, 4346)


## 4. Tuning K — 5-fold CV chỉ trên `train.csv`, chống rò rỉ bằng Pipeline (Checklist E)

Quy tắc quan trọng nhất: **không nhìn `test.csv` khi chọn K**.

Cách làm đúng (đã sửa theo góp ý): vectorizer + KNN được gói trong
`sklearn.pipeline.Pipeline` và `cross_val_score` chạy **trên văn bản đã preprocessing**.
`cross_val_score` clone pipeline cho từng fold nên mỗi fold **tự fit vectorizer**
(từ vựng / trọng số IDF) chỉ trên phần train của fold đó — fold validation không
rò rỉ thông tin vào bước vectorization.

Mỗi cặp (feature, k) được chấm bằng Stratified 5-Fold CV trên toàn bộ train;
điểm chọn model là **F1 trung bình của lớp Spam (lớp dương)**.

KNN dùng `metric="cosine"`, `algorithm="brute"` (vector thưa chiều cao, brute-force vẫn nhanh với ~2.4k mẫu).

In [6]:
%%time
# Minh họa cấu trúc pipeline dùng trong CV (mỗi fold tự fit vectorizer):
print(build_knn_pipeline(build_tfidf_vectorizer, 3))
print()

tune_rows = tune_k_cross_val(train_text, y_train)

tune_df = pd.DataFrame(tune_rows)
tune_df

Pipeline(steps=[('vectorizer', TfidfVectorizer(max_features=5000)),
                ('knn',
                 KNeighborsClassifier(algorithm='brute', metric='cosine',
                                      n_jobs=-1, n_neighbors=3))])



[Count] k= 3  f1=0.9128 ± 0.0226  acc=0.9557


[Count] k= 5  f1=0.9031 ± 0.0199  acc=0.9511


[Count] k= 7  f1=0.8896 ± 0.0238  acc=0.9449


[Count] k= 9  f1=0.8757 ± 0.0307  acc=0.9390


...

[TFIDF] k= 9  f1=0.8351 ± 0.0301  acc=0.9211


[TFIDF] k=11  f1=0.8300 ± 0.0257  acc=0.9190


[TFIDF] k=15  f1=0.8072 ± 0.0376  acc=0.9102


[TFIDF] k=21  f1=0.7768 ± 0.0339  acc=0.8989
CPU times: total: 1min 37s
Wall time: 1min 31s


,feature,k,f1_mean,f1_std,accuracy_mean
0,Count,3,0.912826,0.022637,0.955724
1,Count,5,0.903079,0.019939,0.951129
2,Count,7,0.889560,0.023763,0.944866
3,Count,9,0.875661,0.030651,0.939016
4,Count,11,0.864967,0.033282,0.934420
5,Count,15,0.834330,0.033316,0.921473
6,Count,21,0.815396,0.045294,0.914788
7,TFIDF,3,0.900043,0.012665,0.949454
8,TFIDF,5,0.881540,0.012871,0.941519
9,TFIDF,7,0.849631,0.029254,0.926901


In [7]:
best_k_by_feature = {}
for feature_name in ("Count", "TFIDF"):
    k, f1 = pick_best_k(tune_rows, feature_name)
    best_k_by_feature[feature_name] = k
    print(f"Best K cho {feature_name}: k={k} (CV F1 = {f1:.4f})")
best_k_by_feature

Best K cho Count: k=3 (CV F1 = 0.9128)
Best K cho TFIDF: k=3 (CV F1 = 0.9000)


{'Count': 3, 'TFIDF': 3}

### Biểu đồ K vs F1

Nhận xét: F1 giảm đơn điệu khi K tăng với cả 2 feature — lớp Spam là thiểu số (~26.7% train),
K lớn làm phiếu bầu của lớp Ham chiếm ưu thế và kéo Recall của Spam xuống. **Best K = 3** cho cả Count lẫn TF-IDF.

So với cách tuning cũ (fit vectorizer một lần trên toàn train rồi mới CV): số của **Count gần như không đổi**
vì từ vựng đầy đủ chỉ có 4.346 token (< `max_features=5000`) nên từ vựng từng fold trùng nhau;
**TF-IDF thay đổi nhẹ** (F1 k=3: 0.8992 → 0.9000) vì trọng số IDF phụ thuộc tần suất tài liệu của từng fold.

In [8]:
plot_k_chart(tune_rows, RESULTS_DIR / "knn_k_chart.png")

df_chart = tune_df.pivot(index="k", columns="feature", values="f1_mean")
ax = df_chart.plot(marker="o", figsize=(7, 4.2), grid=True)
ax.set_xlabel("K (n_neighbors)")
ax.set_ylabel("CV F1 (train.csv)")
ax.set_title("KNN: K vs F1 (5-fold CV)")
plt.show()

Saved: C:\email_spam\.claude\worktrees\knn-finish\results\knn\knn_k_chart.png


C:\Users\mac\AppData\Local\Temp\ipykernel_12952\3743757883.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Train cuối + đánh giá trên `test.csv` (Checklist F + G)

Với mỗi feature: refit vectorizer trên toàn bộ train → train KNN với best K (đo **training time**)
→ predict trên test (đo **prediction time**) → tính Accuracy / Precision / Recall / F1 + Confusion Matrix.

`test.csv` chỉ được chạm vào đúng một lần ở cell này.

In [9]:
rows = []
error_frames = {}
metrics_summary = {"n_test_samples": int(len(test))}
x_by_feature = {}

for feature_name, builder in [("Count", build_count_vectorizer), ("TFIDF", build_tfidf_vectorizer)]:
    k = best_k_by_feature[feature_name]
    vectorizer = builder()
    x_train, x_test = fit_transform_train_test(vectorizer, train_text, test_text)
    x_by_feature[feature_name] = (x_train, x_test)

    model = build_knn(k)
    preds, train_time, pred_time = timed_fit_predict(model, x_train, y_train, x_test)

    metrics = classification_metrics(y_test, preds)
    row = result_row("KNN", feature_name, metrics, train_time, pred_time)
    row["k"] = k
    rows.append(row)
    metrics_summary[feature_name] = {"k": k, "metrics": metrics}
    error_frames[feature_name] = _error_frame(test, test_text, preds)

    print(f"KNN + {feature_name} (k={k}): acc={metrics['accuracy']:.4f} "
          f"prec={metrics['precision']:.4f} rec={metrics['recall']:.4f} f1={metrics['f1']:.4f} "
          f"| train={train_time*1000:.1f}ms predict={pred_time*1000:.1f}ms")

results_df = pd.DataFrame(rows)
results_df

KNN + Count (k=3): acc=0.9397 prec=0.8631 rec=0.9177 f1=0.8896 | train=4.2ms predict=268.9ms


KNN + TFIDF (k=3): acc=0.9296 prec=0.8537 rec=0.8861 f1=0.8696 | train=5.2ms predict=255.4ms


,model,feature,accuracy,precision,recall,f1,training_time,prediction_time,k
0,KNN,Count,0.939698,0.863095,0.917722,0.889571,0.004245,0.268884,3
1,KNN,TFIDF,0.929648,0.853659,0.886076,0.869565,0.005179,0.255426,3


### Confusion Matrix cho 2 cấu hình

In [10]:
for feature_name in ("Count", "TFIDF"):
    k = best_k_by_feature[feature_name]
    cm = metrics_summary[feature_name]["metrics"]["confusion_matrix"]
    plot_confusion_matrix(
        cm,
        title=f"KNN + {feature_name} (k={k}) — Confusion Matrix",
        out_path=RESULTS_DIR / f"confusion_matrix_knn_{feature_name.lower()}.png",
    )
    print(cm, "\n")

Saved: C:\email_spam\.claude\worktrees\knn-finish\results\knn\confusion_matrix_knn_count.png
[[416  23]
 [ 13 145]] 



Saved: C:\email_spam\.claude\worktrees\knn-finish\results\knn\confusion_matrix_knn_tfidf.png
[[415  24]
 [ 18 140]] 



## 6. Error Analysis (Checklist G)

Với **KNN + Count (k=3)**: 23 FP + 13 FN trên 597 mẫu test.

- **False Positive** (Ham → đoán Spam): tin nhắn hợp lệ nhưng mang "văn phong quảng cáo" — khuyến mãi, mã giảm giá, thông báo đơn hàng kèm link/`[MONEY]`. Hàng xóm cosine gần nhất của chúng lẫn nhiều tin Spam thật.
- **False Negative** (Spam → đoán Ham): tin Spam dài kiểu "đe dọa pháp lý / nhắc nợ" dùng ngôn ngữ hành chính giống tin nhắn nhà mạng, hoặc spam cờ bạc ngắn chèn URL lạ (tách token hiếm) nên rơi vào vùng toàn Ham.

Với **TF-IDF (k=3)**: 24 FP + 18 FN — nhiều FN hơn vì TF-IDF hạ trọng số token phổ biến (`quy khach`, `thong bao`...), trong khi chính các token này xuất hiện dày trong spam lừa đảo tiếng Việt.

Ghi file `error_analysis_knn.md` (5 FP + 5 FN mỗi cấu hình):

In [11]:
write_error_analysis_md(
    error_frames, metrics_summary, RESULTS_DIR / "error_analysis_knn.md"
)

for feature_name, frame in error_frames.items():
    print(f"===== KNN + {feature_name} =====")
    print("--- 5 False Positive (Ham đoán Spam) ---")
    for msg in frame[frame["error_type"] == "FP"]["message"].head(5):
        print(" •", " ".join(str(msg).split())[:130])
    print("--- 5 False Negative (Spam đoán Ham) ---")
    for msg in frame[frame["error_type"] == "FN"]["message"].head(5):
        print(" •", " ".join(str(msg).split())[:130])
    print()

Saved: C:\email_spam\.claude\worktrees\knn-finish\results\knn\error_analysis_knn.md
===== KNN + Count =====
--- 5 False Positive (Ham đoán Spam) ---
 • Công ty điện lực thủ đức đã tiếp nhận thông tin của quý khách ! ĐL đã cử a Hoàng khảo sát với số ĐT :[NUMBER] khảo sát ngày [DATE]
 • Mung sinh nhat 15 tuoi Lotte Cinema tang ban 2 code xem phim [MONEY] chi voi [MONEY].Dung tai rap tu 10-13/4,17-20/4,24-27/4. Code
 • Ahamove tang rieng ban ma HIAHA tiet kiem ngay [MONEY] cho 2 don hang dau tien, dat don ngay tai https://ahamove.onelink.me/fmLb/5
 • The Manson thong bao, don hang: 221007R4FP4BRJ da duoc kich hoat bao hanh dien tu. Chuc quy khach co giay phut trai nghiem that to
 • Tôi đã nhận được đơn hàng spxvn[NUMBER]a Vào lúc 13h10 ngày 6 thang 10 . Đồng ý gở khieu nai
--- 5 False Negative (Spam đoán Ham) ---
 • [QC]QK du dieu kien vay tieu dung den [MONEY] VND, ls uu dai tu EASY CREDIT (EVN Finance). Soan Y gui [NUMBER] de DK vay va dong y
 • TT: [NUMBER].G@me bài tra thu0ng kie'mtie

## 7. Lưu kết quả (Checklist H)

Xuất `results/knn/results_knn.csv` và `results/knn/knn_k_comparison.csv` — đúng schema yêu cầu:
`model,feature,accuracy,precision,recall,f1,training_time,prediction_time,k`.

In [12]:
results_path = RESULTS_DIR / "results_knn.csv"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
results_df.to_csv(results_path, index=False)
tune_df.to_csv(RESULTS_DIR / "knn_k_comparison.csv", index=False)
print("Saved:", results_path)
print("Saved:", RESULTS_DIR / "knn_k_comparison.csv")

results_df

Saved: C:\email_spam\.claude\worktrees\knn-finish\results\knn\results_knn.csv
Saved: C:\email_spam\.claude\worktrees\knn-finish\results\knn\knn_k_comparison.csv


,model,feature,accuracy,precision,recall,f1,training_time,prediction_time,k
0,KNN,Count,0.939698,0.863095,0.917722,0.889571,0.004245,0.268884,3
1,KNN,TFIDF,0.929648,0.853659,0.886076,0.869565,0.005179,0.255426,3


## 8. Kết luận phần KNN

| Cấu hình | Accuracy | Precision | Recall | F1 | Train time | Predict time |
|---|---:|---:|---:|---:|---:|---:|
| KNN + Count (k=3) | **0.9397** | **0.8631** | **0.9177** | **0.8896** | ~2 ms | ~150 ms |
| KNN + TF-IDF (k=3) | 0.9296 | 0.8537 | 0.8861 | 0.8696 | ~3 ms | ~150 ms |

1. **Best K = 3** cho cả hai feature — chọn bằng 5-fold CV **Pipeline (vectorizer fit trong từng fold)** trên train, không dùng test, không rò rỉ dữ liệu.
2. **CountVectorizer > TF-IDF** với KNN-cosine trên dữ liệu này (F1 0.8896 vs 0.8696): vector đếm giữ tín hiệu lặp của token spam (`[MONEY]`, `nhan`, `dang ky`...) mà TF-IDF làm mờ.
3. KNN "train" gần như tức thời (lazy learner, chỉ lưu ma trận) nhưng **predict chậm hơn NB đáng kể** vì phải tính khoảng cách tới toàn bộ 2.394 mẫu train — chi phí tăng tuyến tính theo kích thước tập train.
4. Recall lớp Spam ~0.92 (Count) là điểm mạnh: KNN bắt được 92% tin lừa đảo; lỗi FN chủ yếu là spam kiểu đe dọa pháp lý có ngôn ngữ giống tin nhắn chính thống.
5. **Dữ liệu có duplicate** (303 dòng train, 31 dòng test theo `message,label`) — giữ nguyên theo dataset chính thức của nhóm, không tự xóa/chia lại; mọi model đều dùng chung một bản nên so sánh vẫn công bằng.
6. Hướng cải thiện: n-gram (1,2), chuẩn hóa teencode, hoặc giảm chiều bằng SVD trước KNN.

Các artifact đã sinh ra trong `results/knn/`:
`results_knn.csv`, `knn_k_comparison.csv`, `knn_k_chart.png`,
`confusion_matrix_knn_count.png`, `confusion_matrix_knn_tfidf.png`, `error_analysis_knn.md`.